In [ ]:
# Bundler kernel: download Qwen3.6-35B-A3B BF16 from HuggingFace,
# then upload as a private Kaggle Dataset for offline mounting in subsequent
# kernels. Run-once. enable_internet=true is required (HF download).
#
# Sequence:
#   1. snapshot_download(Qwen/Qwen3.6-35B-A3B) -> /tmp/qwen36_bf16/  (~70 GB)
#   2. write dataset-metadata.json
#   3. kaggle datasets create -p <dir>  (or kagglehub.dataset_upload)
#
# Time budget on Kaggle (~gigabit network):
#   download : ~20-40 min
#   upload   : ~20-40 min
#   total    : 40-80 min, well under the 12h kernel cap.
import os, json, sys, time, subprocess
from pathlib import Path

HF_REPO       = 'Qwen/Qwen3.6-35B-A3B'
DATASET_OWNER = os.environ.get('KAGGLE_USER', 'cataluna84')
DATASET_SLUG  = 'qwen3-6-35b-a3b-bf16'
DATASET_TITLE = 'Qwen3.6-35B-A3B BF16 (HF mirror for ARC-AGI-3)'
LOCAL_DIR     = '/tmp/qwen36_bf16'

Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)

# --- Step 1: HF download (safetensors + tokenizer + config only) -------------
from huggingface_hub import snapshot_download
print(f'[1/3] downloading {HF_REPO} -> {LOCAL_DIR}', flush=True)
t0 = time.time()
snapshot_download(
    HF_REPO,
    local_dir=LOCAL_DIR,
    allow_patterns=[
        '*.json', '*.txt', '*.md', '*.py',
        'model-*.safetensors', 'model.safetensors',
        'model.safetensors.index.json',
        'tokenizer*', 'preprocessor*', 'chat_template*',
        'generation_config.json', 'special_tokens_map.json',
    ],
    ignore_patterns=[
        '*.bin', '*.fp32.*', '*.gguf', '*.onnx', '*.h5',
        '*.msgpack', '*.tflite',
    ],
)
dl_min = (time.time() - t0) / 60.0
total_bytes = sum(f.stat().st_size for f in Path(LOCAL_DIR).rglob('*') if f.is_file())
print(f'[1/3] done in {dl_min:.1f} min  total={total_bytes/1e9:.2f} GB', flush=True)
print('files:')
for f in sorted(Path(LOCAL_DIR).rglob('*')):
    if f.is_file():
        print(f'  {f.stat().st_size/1e9:>6.2f} GB  {f.relative_to(LOCAL_DIR)}')

# --- Step 2: write dataset-metadata.json ------------------------------------
meta = {
    'title': DATASET_TITLE,
    'id': f'{DATASET_OWNER}/{DATASET_SLUG}',
    'licenses': [{'name': 'apache-2.0'}],
    'keywords': ['llm', 'qwen', 'moe', 'arc-agi', 'arc-prize-2026'],
    'subtitle': 'Qwen3.6-35B-A3B BF16 weights, mirrored from HF for offline Kaggle eval',
    'description': (
        'Verbatim mirror of Qwen/Qwen3.6-35B-A3B (Apache 2.0). '
        'Bundled here so kernels with enable_internet=false can mount '
        'the safetensors at /kaggle/input/<slug>/.'
    ),
    'isPrivate': True,
}
with open(Path(LOCAL_DIR) / 'dataset-metadata.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print('[2/3] wrote dataset-metadata.json', flush=True)

# --- Step 3: try kagglehub first, fall back to kaggle CLI -------------------
print('[3/3] uploading to Kaggle Datasets...', flush=True)
uploaded = False
t1 = time.time()
try:
    import kagglehub
    handle = f'{DATASET_OWNER}/{DATASET_SLUG}'
    print(f'    trying kagglehub.dataset_upload({handle!r}, ...)', flush=True)
    h = kagglehub.dataset_upload(handle, LOCAL_DIR, version_notes='Qwen3.6-35B-A3B bf16 initial bundle')
    print(f'    kagglehub returned: {h!r}', flush=True)
    uploaded = True
except Exception as e:
    print(f'    kagglehub.dataset_upload failed: {type(e).__name__}: {e}', flush=True)

if not uploaded:
    print('    falling back to kaggle CLI: datasets create -p ...', flush=True)
    res = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', LOCAL_DIR, '--dir-mode', 'tar'],
        capture_output=True, text=True, timeout=7200,
    )
    print('CLI stdout:', res.stdout)
    print('CLI stderr:', res.stderr)
    print('CLI rc    :', res.returncode)
    if res.returncode == 0:
        uploaded = True

ul_min = (time.time() - t1) / 60.0
print(f'[3/3] upload finished in {ul_min:.1f} min  success={uploaded}', flush=True)
print()
print('=== SUMMARY ===')
print(f'  HF source        : {HF_REPO}')
print(f'  local cache      : {LOCAL_DIR}')
print(f'  total bytes      : {total_bytes/1e9:.2f} GB')
print(f'  download time    : {dl_min:.1f} min')
print(f'  upload time      : {ul_min:.1f} min')
print(f'  Kaggle Dataset   : {DATASET_OWNER}/{DATASET_SLUG}')
print(f'  uploaded ok      : {uploaded}')
